In [0]:
table_name = "dim_icd10_codes"
silver_table_fqn = f"hdfc_ergo.hdfc_silver.{table_name}"

df = spark.read.table("hdfc_ergo.hdfc_bronze.dim_icd10_codes")
display(df.limit(5))

In [0]:
from pyspark.sql.functions import upper, trim, col, when

# Standardize text: UPPER(TRIM()) for text columns
df = df.withColumn("icd10_code", upper(trim(col("icd10_code"))))
df = df.withColumn("icd10_description", upper(trim(col("icd10_description"))))
df = df.withColumn("category", upper(trim(col("category"))))
df = df.withColumn("severity", upper(trim(col("severity"))))

# Boolean Optimization for is_preventable: Map 'yes'/'1' -> TRUE, 'no'/'0' -> FALSE
df = df.withColumn("is_preventable", 
                   when(upper(trim(col("is_preventable"))).isin("YES", "1"), True)
                   .when(upper(trim(col("is_preventable"))).isin("NO", "0"), False)
                   .otherwise(None).cast("boolean"))

# Drop audit columns
df = df.drop("_ingested_at", "_source_file")

# Also dropping the source 'icd10_code_sk' because we will generate our own Surrogate Key in Gold
df = df.drop("icd10_code_sk")

display(df.limit(5))

In [0]:
df.write.mode("overwrite") \
  .option("overwriteSchema", "true") \
  .saveAsTable(silver_table_fqn)

print(f"✅ Successfully wrote {table_name} to Silver layer.")